# KSP Customer Segmentation

ML clustering model built from 867 companies × 72 features.

**Data source:** `data/companies/company_features.csv` (generated by `scripts/enrich_companies.py`)

**Goal:** Identify customer segments to target new customers and drive marketing strategy.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", 80)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

DATA_PATH = Path("../data/companies/company_features.csv")
df = pd.read_csv(DATA_PATH)
print(f"Dataset: {df.shape[0]} companies × {df.shape[1]} features")
df.head()

In [ ]:
# Feature completeness
completeness = df.notna().mean().sort_values(ascending=False)
print("Feature completeness (showing <100%):")
partial = completeness[completeness < 1.0]
for col, pct in partial.items():
    print(f"  {col:40s} {pct:5.1%}")

## 1. Feature Selection & Preprocessing

In [ ]:
# Select numeric features for clustering
# Exclude identifiers, labels, and derived categorical columns
exclude_cols = [
    "company", "value_quintile", "frequency_tier",
    "is_active_12m", "is_active_6m", "is_churned",
    "peak_month",  # categorical
    # External features (if present)
    "ch_company_name", "company_number", "company_type",
    "company_status", "sic_codes", "industry_sector",
    "incorporation_date", "region", "accounts_type",
]

feature_cols = [c for c in df.columns if c not in exclude_cols and df[c].dtype in ["float64", "int64", "Int64"]]
print(f"Clustering features: {len(feature_cols)}")
for c in feature_cols:
    print(f"  {c}")

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

X_raw = df[feature_cols].copy()

# Impute missing values with median
imputer = SimpleImputer(strategy="median")
X_imputed = pd.DataFrame(imputer.fit_transform(X_raw), columns=feature_cols)

# Log-transform skewed monetary features
log_cols = [c for c in feature_cols if any(kw in c for kw in ["monetary", "revenue", "cost", "price", "quantity"])]
for c in log_cols:
    X_imputed[c] = np.log1p(X_imputed[c].clip(lower=0))

# Standardise
scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X_imputed), columns=feature_cols)

print(f"Preprocessed: {X_scaled.shape}")
X_scaled.describe().round(2)

## 2. Dimensionality Reduction (PCA)

In [ ]:
from sklearn.decomposition import PCA

# Determine optimal PCA components (95% variance)
pca_full = PCA().fit(X_scaled)
cumvar = np.cumsum(pca_full.explained_variance_ratio_)
n_components_95 = np.argmax(cumvar >= 0.95) + 1
n_components_90 = np.argmax(cumvar >= 0.90) + 1

print(f"Components for 90% variance: {n_components_90}")
print(f"Components for 95% variance: {n_components_95}")
print(f"\nTop 15 components explained variance:")
for i, (var, cum) in enumerate(zip(pca_full.explained_variance_ratio_[:15], cumvar[:15])):
    bar = "#" * int(var * 200)
    print(f"  PC{i+1:2d}: {var:5.1%}  (cum: {cum:5.1%})  {bar}")

In [ ]:
# Apply PCA with chosen components
n_comp = min(n_components_95, 20)  # cap at 20 for interpretability
pca = PCA(n_components=n_comp)
X_pca = pca.fit_transform(X_scaled)
print(f"PCA: {X_pca.shape[1]} components, {pca.explained_variance_ratio_.sum():.1%} variance")

## 3. Optimal Cluster Count

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

K_range = range(2, 12)
inertias = []
silhouettes = []

for k in K_range:
    km = KMeans(n_clusters=k, n_init=10, random_state=42)
    labels = km.fit_predict(X_pca)
    inertias.append(km.inertia_)
    sil = silhouette_score(X_pca, labels)
    silhouettes.append(sil)
    print(f"  k={k:2d}  inertia={km.inertia_:,.0f}  silhouette={sil:.3f}")

best_k = K_range[np.argmax(silhouettes)]
print(f"\nBest k by silhouette: {best_k}")

## 4. Final Clustering

In [ ]:
# Fit final model with best k
km_final = KMeans(n_clusters=best_k, n_init=20, random_state=42)
df["cluster"] = km_final.fit_predict(X_pca)

print(f"Cluster sizes:")
for c, count in df["cluster"].value_counts().sort_index().items():
    print(f"  Cluster {c}: {count} companies ({100*count/len(df):.1f}%)")

In [ ]:
# Cluster profiles - key metrics
profile_cols = [
    "frequency", "monetary_total", "monetary_mean", "recency_days",
    "avg_quantity", "avg_unit_price", "avg_margin", "avg_num_operations",
    "product_type_diversity", "orders_per_year", "tenure_days",
    "revenue_trend_pct", "recent_share_pct",
]
available = [c for c in profile_cols if c in df.columns]

profiles = df.groupby("cluster")[available].agg(["mean", "median"]).round(2)
profiles

In [ ]:
# Product type preferences by cluster
ptype_cols = [c for c in df.columns if c.startswith("ptype_")]
ptype_profiles = df.groupby("cluster")[ptype_cols].mean().round(1)
ptype_profiles.columns = [c.replace("ptype_", "").replace("_pct", "") for c in ptype_profiles.columns]
print("Product type preferences by cluster (%):")
ptype_profiles

In [ ]:
# Operations preferences by cluster
op_cols = [c for c in df.columns if c.startswith("op_")]
op_profiles = df.groupby("cluster")[op_cols].mean().round(1)
op_profiles.columns = [c.replace("op_", "").replace("_pct", "") for c in op_profiles.columns]
print("Operations usage by cluster (%):")
op_profiles

## 5. Segment Naming & Interpretation

In [ ]:
# Automated segment naming heuristic based on cluster characteristics
cluster_summary = df.groupby("cluster").agg(
    n=("company", "count"),
    avg_revenue=("monetary_total", "mean"),
    avg_frequency=("frequency", "mean"),
    avg_recency=("recency_days", "mean"),
    avg_quantity=("avg_quantity", "mean"),
    avg_operations=("avg_num_operations", "mean"),
    avg_margin=("avg_margin", "mean"),
).round(1)

print("Cluster summary for naming:")
cluster_summary

In [ ]:
# Top companies in each cluster
for c in sorted(df["cluster"].unique()):
    cluster_df = df[df["cluster"] == c].nlargest(5, "monetary_total")
    names = cluster_df["company"].tolist()
    revenue = cluster_df["monetary_total"].tolist()
    print(f"\nCluster {c} - Top 5 by revenue:")
    for name, rev in zip(names, revenue):
        print(f"  £{rev:>10,.2f}  {name}")

## 6. Export Segmented Dataset

In [ ]:
# Save final segmented dataset
output_path = Path("../data/companies/company_segments.csv")
df.to_csv(output_path, index=False)
print(f"Saved segmented dataset: {output_path}")
print(f"Shape: {df.shape}")
print(f"\nColumns: {list(df.columns)}")